# Face Recognition: Training and Testing
This notebook reads the images from the `authorized_faces` directory, assigns IDs to each person, trains an LBPH Face Recognizer, and tests it.

In [8]:
import cv2
import numpy as np
import os
import pickle
import matplotlib.pyplot as plt

# Use matplotlib to show images inline in the notebook
def show_image(img, title="Image"):
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()

## 1. Data Preparation & Training
Map names to IDs and train the OpenCV LBPH model.

In [9]:
data_dir = 'authorized_faces'

name_dict = {}
current_id = 0

faces = []
labels = []

# 1. Read the dataset
for person_name in os.listdir(data_dir):
    person_path = os.path.join(data_dir, person_name)
    
    if not os.path.isdir(person_path):
        continue
        
    # Assign an ID
    if person_name not in name_dict:
        name_dict[current_id] = person_name # Note: ID as key, Name as value for easier lookup later
        current_id += 1
        
    label_id = [k for k, v in name_dict.items() if v == person_name][0]
    
    # 2. Extract faces and labels
    for image_name in os.listdir(person_path):
        if image_name.endswith("jpg") or image_name.endswith("png"):
            img_path = os.path.join(person_path, image_name)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            
            if img is not None:
                faces.append(img)
                labels.append(label_id)

print(f"Names mapped to IDs: {name_dict}")
print(f"Total faces loaded: {len(faces)}")

# 3. Train the model
print("Training the LBPH Face Recognizer...")
recognizer = cv2.face.LBPHFaceRecognizer_create()
recognizer.train(faces, np.array(labels))

# 4. Save the model and the dictionary
recognizer.write('face_model_trained.yml')
with open('name_dict.pkl', 'wb') as f:
    pickle.dump(name_dict, f)

print("✅ Training complete! Saved as 'face_model_trained.yml' and 'name_dict.pkl'")

Names mapped to IDs: {0: 'chaitanya', 1: 'harsh', 2: 'rashid', 3: 'saish'}
Total faces loaded: 10
Training the LBPH Face Recognizer...
✅ Training complete! Saved as 'face_model_trained.yml' and 'name_dict.pkl'


## 2. Testing the Model
Let's run a test on a specific image to see if the model correctly identifies the face and draws a bounding box.

In [10]:
# Load the trained model and dictionary
recognizer = cv2.face.LBPHFaceRecognizer_create()
recognizer.read('face_model_trained.yml')

with open('name_dict.pkl', 'rb') as f:
    loaded_dict = pickle.load(f)

face_cascade = cv2.CascadeClassifier('face_model.xml')

# --- TEST IMAGE ---
# Change this path to an actual image you want to test! 
# You can grab one from your authorized_faces folder or a completely new image.
test_img_path = 'authorized_faces/chaitanya/1.jpg' 

img = cv2.imread(test_img_path)
if img is None:
    print(f"Error: Could not load image at {test_img_path}")
else:
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    detected_faces = face_cascade.detectMultiScale(gray, scaleFactor=1.2, minNeighbors=5)

    for (x, y, w, h) in detected_faces:
        # Extract the face region
        roi_gray = gray[y:y+h, x:x+w]
        
        # Predict the face
        id_, confidence = recognizer.predict(roi_gray)
        
        # Confidence logic (Lower is better in LBPH, usually < 100 is a good match)
        if confidence < 85:
            name = loaded_dict[id_]
            color = (0, 255, 0) # Green for authorized
            text = f"{name} ({round(100 - confidence)}%)"
        else:
            name = "UNKNOWN"
            color = (0, 0, 255) # Red for unauthorized/suspicious
            text = f"{name} ({round(100 - confidence)}%)"
            
        # Draw bounding box and label
        cv2.rectangle(img, (x, y), (x+w, y+h), color, 2)
        cv2.putText(img, text, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    show_image(img, title="Prediction Result")

Error: Could not load image at authorized_faces/chaitanya/1.jpg
